# Stage 04 — Build and validate the 2022 transmission topology

This stage reconstructs the mainland SNI transmission boundary from the checksum-pinned PyPSA-Earth base grid. It supports 138, 230, and 500 kV, preserves every mutation in an audit table, and does not solve the network. The five added corridors and transformer-resistance rule are legacy modeling decisions, not official Master Plan facts.

In [1]:
from pathlib import Path
import json
import sys
import matplotlib.pyplot as plt
import pandas as pd
import pypsa

def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'src').is_dir() and (candidate / 'config/artifacts.yaml').is_file():
            return candidate
    raise RuntimeError('Could not locate the EcuadorElectricGrid repository root.')

ROOT = find_repository_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.artifacts import format_preflight_report, run_preflight
from src.topology import build_2022_topology, load_repair_table
from src.topology_plots import save_phase05_topology_maps

preflight = run_preflight('04_topology', root=ROOT, offline=True)
print(format_preflight_report(preflight))
preflight.raise_for_errors()


Artifact preflight: PASS (stage=04_topology)
  valid        pypsa_earth_base_ec: cached file verified
  not-required renewable_profile_solar_ec: stage does not require this artifact
  valid        gadm_ecuador_41: cached file verified
  not-required renewable_profile_onwind_ec: stage does not require this artifact
  not-required demand_template_2013_ec: stage does not require this artifact
  not-required cenace_sni_2022_monthly_demand: stage does not require this artifact
  not-required cenace_2022_net_supply: stage does not require this artifact
  not-required generation_existing_2022_source: stage does not require this artifact
  not-required generation_location_evidence_2022: stage does not require this artifact
  not-required generation_location_overrides_2022: stage does not require this artifact
  not-required cenace_2022_commissioning_events: stage does not require this artifact
  not-required generation_fleet_2022_normalized: stage does not require this artifact
  not-required 

## Inputs and transformation contract

The base network is an upstream input. The repair table is a small reviewed interface that records rationale, evidence location, confidence, and whether each row is an official fact. Voltage snapping changes 46 kV to 48 kV and 145 kV to 138 kV before the lower-voltage topology is removed.

In [2]:
base_path = ROOT / 'data/raw/networks/base.nc'
repair_path = ROOT / 'data/interim/network/topology_repairs_2022.csv'
base = pypsa.Network(base_path)
repairs = load_repair_table(repair_path)
result = build_2022_topology(base, repairs)
result.fingerprint


INFO:pypsa.io:Imported network base.nc has buses, lines, transformers


{'snapshots': 8760,
 'buses': 110,
 'lines': 115,
 'transformers': 19,
 'loads': 0,
 'generators': 0,
 'bus_voltage_counts': {'138.0': 64, '230.0': 40, '500.0': 6},
 'connected_components': 2}

## Diagnostics and evidence

The mutation audit distinguishes deterministic filtering from reviewed repairs. Connected components are reported rather than silently bridged: the retained topology has one main component and any smaller upstream islands remain visible for Phase 6 feasibility diagnosis. The maps retain the project's established voltage palette and provincial-boundary styling, with one combined map and separate bus-labeled maps for 500, 230, and 138 kV.

In [3]:
audit_dir = ROOT / 'data/interim/network'
figure_dir = ROOT / 'reports/figures/phase_05'
network_dir = ROOT / 'data/processed/networks'
for directory in (audit_dir, figure_dir, network_dir):
    directory.mkdir(parents=True, exist_ok=True)
result.mutation_audit.to_csv(audit_dir / 'topology_mutation_audit_2022.csv', index=False)
result.component_summary.to_csv(audit_dir / 'topology_components_2022.csv', index=False)
result.voltage_contraction_audit.to_csv(audit_dir / 'low_voltage_bridge_audit_2022.csv', index=False)
with (audit_dir / 'topology_fingerprint_2022.json').open('w', encoding='utf-8') as handle:
    json.dump(result.fingerprint, handle, indent=2, sort_keys=True)

topology_figures = save_phase05_topology_maps(
    result.network,
    ROOT / 'data/raw/gadm/gadm41_ECU.gpkg',
    figure_dir,
)
plt.show()
for figure in topology_figures.values():
    plt.close(figure)


C:\Users\fpenaherrera_vaca\AppData\Local\Temp\ipykernel_10660\116496078.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Candidate topology artifact

The NetCDF below is a generated intermediate product. It becomes an input to Stage 05 but is not authoritative or validated scientific output. Stage 06 must still establish successful optimization and the complete energy/statistical acceptance gate.

In [4]:
topology_path = network_dir / 'ec_network_2022.nc'
result.network.export_to_netcdf(topology_path)
reloaded = pypsa.Network(topology_path)
assert len(reloaded.buses) == result.fingerprint['buses']
assert len(reloaded.lines) == result.fingerprint['lines']
assert len(reloaded.transformers) == result.fingerprint['transformers']
print(f'Wrote candidate topology: {topology_path.relative_to(ROOT)}')


INFO:pypsa.io:Exported network 'ec_network_2022.nc' contains: carriers, lines, transformers, buses


INFO:pypsa.io:Imported network ec_network_2022.nc has buses, carriers, lines, transformers


Wrote candidate topology: data\processed\networks\ec_network_2022.nc
